# GP Dictionary Builder & Updater

## Objective

This notebook builds and maintains the master Global Parent (GP) dictionary.

The dictionary is the source of truth used to homologate different spellings of the same company into a single canonical Global Parent.

Example

ABC INC
ABC LLC
ABC CORPORATION

↓

ABC

The notebook supports two execution modes.

### CREATE

Build the first GP dictionary from:

- normalization_pairs.csv
- reviewed algorithm output

### UPDATE

Safely update an existing GP dictionary using:

- existing GP dictionary
- reviewed algorithm output

The dictionary is never rebuilt after it exists.

Every execution either:

- creates the initial dictionary
- or appends new validated mappings.

Before saving, the notebook automatically checks for mapping conflicts to ensure dictionary integrity.

In [ ]:
from __future__ import annotations
from enum import Enum
from pathlib import Path
import pandas as pd

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

class ExecutionMode(Enum):
    CREATE = "create"
    UPDATE = "update"


MODE = ExecutionMode.CREATE

NORMALIZATION_PAIRS_PATH = Path(r"C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\GP Cleaning\Output\normalization_pairs.csv")

REVIEW_PATH = Path(r"C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\GP Cleaning\fuzzy_reviewed.csv")

DICTIONARY_PATH = Path(r"C:\Users\reyiv\Documents\Quaker Houghton\Codigos\limpieza-brandname rep\limpieza-brandname\GP Cleaning\Output\gp_dictionary.csv")

In [ ]:
class DictionaryUpdatePipeline:
    """
    Build and maintain the Global Parent dictionary.
    """

    def __init__(
        self,
        mode: ExecutionMode,
        normalization_pairs_path: Path,
        review_path: Path,
        dictionary_path: Path,
        output_path: Path,
    ) -> None:

        self.mode = mode

        self.normalization_pairs_path = normalization_pairs_path
        self.review_path = review_path
        self.dictionary_path = dictionary_path
        self.output_path = output_path

        # Input DataFrames

        self.normalization_df = pd.DataFrame()

        self.review_df = pd.DataFrame()

        self.dictionary_df = pd.DataFrame()

        # Mapping DataFrames

        self.normalization_mappings_df = pd.DataFrame()

        self.review_mappings_df = pd.DataFrame()

        self.new_mappings_df = pd.DataFrame()

        # Output

        self.updated_dictionary_df = pd.DataFrame()

    def run(self) -> None:

        self._load_inputs()

        self._validate_inputs()

        self._build_review_mappings()

        if self.mode == ExecutionMode.CREATE:

            self._build_normalization_mappings()

            self._create_dictionary()

        else:

            self._update_dictionary()

        self._detect_conflicts()

        self._sort_dictionary()

        self._save_dictionary()

        self._print_summary()